<a href="https://colab.research.google.com/github/geimarcopete/LENGUAJES-DE-PROGRAMACION/blob/main/frontera_eficiente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Frontera Eficiente de Markowitz — Notebook integrado

Este notebook contiene un flujo completo para descargar precios con `yfinance`, calcular retornos,
construir la matriz de covarianza, optimizar portafolios (Máx Sharpe y Mínima Varianza),
simular portafolios mediante Monte Carlo, y graficar la frontera eficiente con **Plotly**.

También incluye un archivo `streamlit_app.py` listo para ejecutar como aplicación Streamlit.

**Instrucciones rápidas**
1. Si estás en Colab: instala dependencias si no están presentes (`yfinance`, `plotly`).
2. Ejecuta las celdas de arriba hacia abajo.
3. Para la app Streamlit, descarga `streamlit_app.py` y ejecútalo localmente o en Streamlit Cloud:
   `streamlit run streamlit_app.py`


In [1]:
# Si estás en Colab, descomenta e instala las dependencias:
# !pip install yfinance plotly streamlit scipy


In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")


In [3]:
# === Parámetros modificables ===
# Puedes editar estos valores directamente en el notebook.
TICKERS_INPUT = "AAPL,MSFT,GOOGL,AMZN"   # tickers separados por coma
START_DATE = "2022-01-01"
END_DATE = "2025-01-01"
RISK_FREE_RATE = 0.04   # 4% en decimal
NUM_PORTFOLIOS = 5000
TRADING_DAYS = 252


In [4]:
# === Descarga de datos ===
tickers = [t.strip().upper() for t in TICKERS_INPUT.split(',') if t.strip()!='']
if len(tickers) == 0:
    raise ValueError("Debe ingresar al menos un ticker válido en TICKERS_INPUT.")

print("Tickers:", tickers)
data = yf.download(tickers, start=START_DATE, end=END_DATE)['Close']

# Si sólo hay un ticker, convertimos a DataFrame con columna
if isinstance(data, pd.Series):
    data = data.to_frame(tickers[0])

print("\nDatos descargados (primeras filas):")
display(data.head())


Tickers: ['AAPL', 'MSFT', 'GOOGL', 'AMZN']


[*********************100%***********************]  4 of 4 completed


Datos descargados (primeras filas):


Ticker,AAPL,AMZN,GOOGL,MSFT
Date,,,,
2022-01-03,178.443100,170.404495,143.998322,324.504547
2022-01-04,176.178391,167.522003,143.410400,318.940277
2022-01-05,171.492096,164.356995,136.831253,306.696899
2022-01-06,168.629303,163.253998,136.803925,304.273254
2022-01-07,168.795975,162.554001,136.078445,304.428406


In [5]:
# === Cálculo de retornos logarítmicos y anualización ===
returns = np.log(data / data.shift(1)).dropna()
mean_returns = returns.mean() * TRADING_DAYS
cov_matrix = returns.cov() * TRADING_DAYS

print("\nRendimientos esperados anualizados (%):")
display((mean_returns * 100).round(2).to_frame(name='Rendimiento Anual (%)'))

print("\nMatriz de covarianza (anualizada):")
display(cov_matrix.round(6))



Rendimientos esperados anualizados (%):


,Rendimiento Anual (%)
Ticker,
AAPL,11.24
AMZN,8.47
GOOGL,9.06
MSFT,8.58



Matriz de covarianza (anualizada):


Ticker,AAPL,AMZN,GOOGL,MSFT
Ticker,,,,
AAPL,0.073153,0.058672,0.055258,0.051126
AMZN,0.058672,0.147044,0.081932,0.073111
GOOGL,0.055258,0.081932,0.107292,0.062730
MSFT,0.051126,0.073111,0.062730,0.076057


In [6]:
# === Funciones auxiliares ===
def portfolio_performance(weights, mean_returns, cov_matrix):
    ret = np.dot(weights, mean_returns)
    vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return ret, vol

def neg_sharpe_ratio(weights, mean_returns, cov_matrix, rf):
    ret, vol = portfolio_performance(weights, mean_returns, cov_matrix)
    return -(ret - rf) / vol

def minimize_volatility(weights, mean_returns, cov_matrix):
    return portfolio_performance(weights, mean_returns, cov_matrix)[1]


In [7]:
# === Optimización (Máx Sharpe y Mínima Varianza) ===
num_assets = len(tickers)
args = (mean_returns, cov_matrix)
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0, 1) for _ in range(num_assets))
initial_guess = num_assets * [1.0 / num_assets]

rf = RISK_FREE_RATE

# Máx Sharpe
max_sharpe = minimize(
    neg_sharpe_ratio,
    initial_guess,
    args=(mean_returns, cov_matrix, rf),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

# Mínima Varianza
min_var = minimize(
    minimize_volatility,
    initial_guess,
    args=(mean_returns, cov_matrix),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

ret_sharpe, vol_sharpe = portfolio_performance(max_sharpe.x, mean_returns, cov_matrix)
ret_min, vol_min = portfolio_performance(min_var.x, mean_returns, cov_matrix)

print("✅ Optimización completada.")


✅ Optimización completada.


In [8]:
# === Simulación Monte Carlo para la frontera eficiente ===
results = np.zeros((3, NUM_PORTFOLIOS))
weights_record = []

for i in range(NUM_PORTFOLIOS):
    weights = np.random.random(num_assets)
    weights /= np.sum(weights)
    weights_record.append(weights)
    portfolio_return = np.dot(weights, mean_returns)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    sharpe_ratio = (portfolio_return - RISK_FREE_RATE) / portfolio_volatility
    results[0, i] = portfolio_volatility
    results[1, i] = portfolio_return
    results[2, i] = sharpe_ratio

results_df = pd.DataFrame(results.T, columns=['Riesgo', 'Retorno', 'Sharpe'])
weights_df = pd.DataFrame(weights_record, columns=tickers)

max_sharpe_idx = results_df['Sharpe'].idxmax()
max_sharpe_port = results_df.loc[max_sharpe_idx]
best_weights = weights_df.loc[max_sharpe_idx]

print("\nPortafolio con mejor Sharpe (simulado):")
print(f"Retorno esperado anual: {max_sharpe_port['Retorno']*100:.2f}%")
print(f"Riesgo (volatilidad): {max_sharpe_port['Riesgo']*100:.2f}%")
print(f"Sharpe Ratio: {max_sharpe_port['Sharpe']:.2f}")
print("\nPesos óptimos (%):")
display(best_weights.apply(lambda x: f"{x*100:.2f}%"))


Portafolio con mejor Sharpe (simulado):
Retorno esperado anual: 11.08%
Riesgo (volatilidad): 26.70%
Sharpe Ratio: 0.27

Pesos óptimos (%):


,1596
AAPL,93.84%
MSFT,3.21%
GOOGL,1.65%
AMZN,1.29%


In [9]:
# === Visualización interactiva con Plotly ===
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=results_df['Riesgo'],
    y=results_df['Retorno'],
    mode='markers',
    marker=dict(
        color=results_df['Sharpe'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='Sharpe Ratio'),
        size=6,
        opacity=0.7
    ),
    name='Portafolios simulados'
))

fig.add_trace(go.Scatter(
    x=[max_sharpe_port['Riesgo']],
    y=[max_sharpe_port['Retorno']],
    mode='markers+text',
    text=['Máx Sharpe (simulado)'],
    textposition='top center',
    marker=dict(color='red', size=12, symbol='star'),
    name='Máx Sharpe'
))

fig.update_layout(
    title='📈 Frontera Eficiente - Simulación Monte Carlo',
    xaxis_title='Riesgo (Volatilidad)',
    yaxis_title='Retorno Esperado Anual',
    template='plotly_white'
)

fig.show()


In [10]:
# === Gráfica de la composición del portafolio óptimo (Pie) ===
fig_pie = go.Figure(data=[go.Pie(labels=tickers, values=max_sharpe.x)])
fig_pie.update_layout(title='Composición del Portafolio Máx Sharpe')
fig_pie.show()


In [11]:
# === Exportar resultados a CSV ===
results_csv = results_df.copy()
results_csv['Riesgo (%)'] = results_csv['Riesgo'] * 100
results_csv['Retorno (%)'] = results_csv['Retorno'] * 100
csv_bytes = results_csv.to_csv(index=False).encode('utf-8')

from IPython.display import HTML
print("Para descargar, en Colab usa la opción de archivos o guarda el archivo localmente:")
with open('frontera_eficiente_results.csv', 'wb') as f:
    f.write(csv_bytes)
print("Archivo 'frontera_eficiente_results.csv' guardado en el directorio actual.")


Para descargar, en Colab usa la opción de archivos o guarda el archivo localmente:
Archivo 'frontera_eficiente_results.csv' guardado en el directorio actual.


---

## Archivo Streamlit incluido

También se ha creado el archivo `streamlit_app.py` en el mismo directorio.  
Puedes ejecutar la app localmente con:

```
streamlit run streamlit_app.py
```

El archivo contiene una versión lista para producción de la app con `st.sidebar`, métricas, gráficos y botón de descarga.
